# 02 - Preprocessing & Class Imbalance (Zero Data Leakage)
Demonstrates: patient-level split, fit-on-train-only, SMOTE inside the Pipeline, and the exclusion of the post-outcome `time` column.
Full logic: `src/preprocessing.py`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fossbk-spec/heart-failure/blob/main/notebooks/02_preprocessing_and_imbalance.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/fossbk-spec/heart-failure/blob/main/notebooks/02_preprocessing_and_imbalance.ipynb)

In [1]:
# Runs offline on Google Colab / Kaggle Notebooks: no internet/Kaggle token needed.
# The dataset CSV is already bundled in the repo at data/raw/.
import os, sys, subprocess
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    if not os.path.exists("heart-failure"):
        subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/fossbk-spec/heart-failure.git"], check=True)
    os.chdir("heart-failure")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the project root locally
sys.path.insert(0, os.getcwd())


In [2]:
from src.data_loader import load_raw_data, split_features_target
from src.preprocessing import patient_level_split, make_full_pipeline, assert_no_fit_on_holdout
from src.models import get_model

df = load_raw_data()
split = patient_level_split(df)
split.summary()

[OK] ZERO DATA LEAKAGE confirmed (Train=191 IDs | Validation=48 IDs | Test=60 IDs).


,split,n_samples,n_patients,n_positive,positive_rate
0,train,191,191,62,0.324607
1,val,48,48,15,0.312500
2,test,60,60,19,0.316667


## Verify Zero Data Leakage (assertion code)

In [3]:
from src.preprocessing import verify_zero_data_leakage
verify_zero_data_leakage(split.train, split.val, split.test)

[OK] ZERO DATA LEAKAGE confirmed (Train=191 IDs | Validation=48 IDs | Test=60 IDs).


True

## Fit the pipeline (Imputer + log1p + Scaler + SMOTE) on Train ONLY

In [4]:
X_train, y_train = split_features_target(split.train)
X_test, y_test = split_features_target(split.test)
print('Features used:', list(X_train.columns))

pipeline = make_full_pipeline(get_model('logistic_regression'))
pipeline.fit(X_train, y_train)
assert_no_fit_on_holdout(pipeline, X_test)  # must pass

Features used: ['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium', 'anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']


True

## Why `time` is excluded: a leakage demonstration
Same grouped 5-fold CV, with and without the follow-up column. The jump in ROC-AUC is NOT real predictive skill — `time` is only known after the outcome has happened.

In [5]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from src import config
from src.preprocessing import cv_splitter

y = df[config.TARGET_COLUMN].astype(int)
groups = df[config.ID_COLUMN]
for label, cols in [('baseline features', config.FEATURE_COLUMNS),
                    ('baseline + time (LEAKY)', config.FEATURE_COLUMNS + ['time'])]:
    rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=42)
    scores = cross_val_score(rf, df[cols], y, groups=groups, cv=cv_splitter(),
                             scoring='roc_auc')
    print(f'{label:<26} ROC-AUC = {scores.mean():.3f} ± {scores.std(ddof=1):.3f}')

baseline features          ROC-AUC = 0.788 ± 0.084
baseline + time (LEAKY)    ROC-AUC = 0.915 ± 0.021


## Save the transformed feature matrix to data/processed/

In [6]:
import pandas as pd
pre = pipeline.named_steps['preprocessor']
processed = pd.DataFrame(pre.transform(X_train), columns=pre.get_feature_names_out())
processed['DEATH_EVENT'] = y_train.to_numpy()
processed.to_csv('data/processed/train_transformed.csv', index=False)
processed.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,anaemia,diabetes,high_blood_pressure,sex,smoking,DEATH_EVENT
0,-0.534518,2.839804,0.002197,0.189364,-0.230145,-0.132738,0.0,0.0,0.0,1.0,0.0,1
1,-0.967971,-0.854641,-1.508695,-0.362299,0.877540,0.115821,1.0,0.0,0.0,1.0,0.0,1
2,0.332390,-0.539271,-1.508695,0.716756,1.713595,-5.103925,1.0,1.0,0.0,0.0,0.0,1
3,1.199298,-0.167346,-1.928387,-1.587713,-0.070499,0.115821,1.0,0.0,0.0,1.0,0.0,1
4,-0.101064,0.046740,1.848843,1.516297,-0.230145,-1.375535,1.0,1.0,0.0,1.0,1.0,1
